# 03 - Neuroglancer Visualization

Interactive visualization of processed tissue sections using Neuroglancer.

**Three viewing modes:**
- **Option A** - OME-Zarr plate view (multi-tissue, one layer per child `.ome.zarr`)
- **Option B** - Precomputed volume (single `precomputed://` URL)
- **Option C** - Shareable state JSON files

This notebook is intentionally separate from notebooks 01, 02, and 04.
If `/output/per_tissue_ngff` is empty, Option A creates a tiny demo NGFF plate automatically.
Option B still expects a precomputed volume if you want to exercise that path.

**Prerequisites:** Docker and the Conda environment already include the visualization stack.
For a local editable install, use `pip install -e ".[visualization]"`.

In [ ]:
from pathlib import Path

from wsi_pipeline.demo_data import create_demo_ngff_plate

from wsi_pipeline.neuroglancer import (
    open_neuroglancer_plate_view,
    open_neuroglancer_precomputed,
    stop_cors_server,
    RGB_SHADER_PLAIN,
    RGB_SHADER,
    R_SHADER,
    G_SHADER,
    B_SHADER,
    emit_ng_state_for_ngff_plate,
    emit_ng_state_for_precomputed_plate,
)

## Option A: View OME-Zarr Plate (Multi-Tissue)

`open_neuroglancer_plate_view()` finds all `*.ome.zarr` (or `*.zarr`) children
in a plate directory, reads physical pixel sizes from NGFF metadata, starts a
CORS + Range HTTP server, and launches Neuroglancer with one layer per tissue.

Set `visible_first_only=True` to avoid visual clutter - only the first layer is
visible by default; toggle others in the Neuroglancer sidebar.

In [ ]:
# This notebook expects a per-tissue NGFF plate directory.
# If none exists, create a tiny demo plate so Option A works out of the box.
plate_root = Path("/output/per_tissue_ngff").expanduser().resolve()

plate_root.mkdir(parents=True, exist_ok=True)
plate_children = sorted(p for p in plate_root.glob("*.ome.zarr") if p.is_dir())
if not plate_children:
    print("No NGFF plate found at /output/per_tissue_ngff. Generating a tiny demo plate...")
    plate_children = create_demo_ngff_plate(plate_root)
    for child in plate_children:
        print(f"  Created {child.name}")

viewer, httpd = open_neuroglancer_plate_view(
    plate_root,
    mode="remote",
    http_host="localhost",
    http_port=8000,
    ng_host="localhost",
    ng_port=9999,
    shader=RGB_SHADER_PLAIN,
    visible_first_only=True,
)
# Paste the printed URL into Chrome or Firefox

In [ ]:
# Optional: embed the viewer directly in the notebook
from IPython.display import IFrame
IFrame(src=viewer.get_viewer_url(), width=1000, height=700)

## Option B: View Precomputed Volume

`open_neuroglancer_precomputed()` accepts a `precomputed://` URL.  When the
URL uses the `file://` scheme, it automatically starts a local HTTP server
and rewrites the URL so the Neuroglancer browser client can access the data.

Supported URL schemes:
- `precomputed://file:///ABS/PATH/volume`
- `precomputed://http(s)://host/path/to/volume`
- `precomputed://gs://bucket/path`
- `precomputed://s3://bucket/path`

In [ ]:
# Stop the previous server first
stop_cors_server(httpd)
httpd = None

precomputed_path = Path("/output/precomputed_plate").expanduser().resolve()
if precomputed_path.exists():
    viewer, httpd = open_neuroglancer_precomputed(
        f"precomputed://file://{precomputed_path.as_posix()}",
        mode="remote",
        http_host="localhost",
        http_port=8000,
        ng_host="localhost",
        ng_port=9999,
    )
else:
    print("No demo precomputed volume found at /output/precomputed_plate.")
    print("Edit this cell to point open_neuroglancer_precomputed() at your own volume.")

## Option C: Generate Shareable State JSON

Write a Neuroglancer state JSON file that any Neuroglancer instance can load.
This is useful for sharing links with collaborators who have access to the
same HTTP-served data.

In [ ]:
# For NGFF plate
emit_ng_state_for_ngff_plate(
    plate_root=plate_root,
    base_http_url="http://localhost:8000/per_tissue_ngff",
    out_state_path=plate_root / "ng_state.json",
)

# For precomputed volume
# emit_ng_state_for_precomputed_plate(
#     precomputed_url="precomputed://http://localhost:8000/precomputed_plate",
#     out_state_path=Path("/output/precomputed_plate/ng_state.json"),
# )

## Shader Options

Available shader constants for RGB tissue images:

| Constant | Description |
|----------|-------------|
| `RGB_SHADER_PLAIN` | Basic `emitRGB(toNormalized(...))` |
| `RGB_SHADER` | Per-channel `#uicontrol invlerp` contrast sliders |
| `R_SHADER` | Red channel only (false-color) |
| `G_SHADER` | Green channel only (false-color) |
| `B_SHADER` | Blue channel only (false-color) |

Pass the desired shader via the `shader=` parameter.

In [ ]:
# Example: re-launch with the red-channel-only shader
stop_cors_server(httpd)

viewer, httpd = open_neuroglancer_plate_view(
    plate_root,
    mode="remote",
    http_host="localhost",
    http_port=8000,
    ng_host="localhost",
    ng_port=9999,
    shader=R_SHADER,
    visible_first_only=True,
)

## Cleanup

In [ ]:
stop_cors_server(httpd)